In [ ]:
def deep_merge(a, b):
    """
    题目描述
    实现函数 deep_merge(a, b)，将两个字典进行深度合并：
    当同一个 key 对应的值都是字典时，需要递归合并。
    当 key 只存在于其中一个字典时，直接保留。
    当同一个 key 对应的值不是两个字典时，以 b 中的值覆盖 a 中的值。
    不允许修改原始输入字典。
    """

    result = {}
    # 合并所有键
    all_keys = set(a.keys()) | set(b.keys())
    for key in all_keys:
        if key in a and key in b:
            val_a, val_b = a[key], b[key]
            if isinstance(val_a, dict) and isinstance(val_b, dict):
                # 两个都是字典，递归合并
                result[key] = deep_merge(val_a, val_b)
            else:
                # 否则以 b 覆盖 a
                result[key] = val_b
        elif key in b:
            result[key] = b[key]
        else:
            result[key] = a[key]
    return result


In [4]:
a = {"db": {"host": "localhost", "port": 3306}, "debug": False}
b = {"db": {"port": 5432}, "debug": True}
print(deep_merge(a, b))


{'debug': True, 'db': {'host': 'localhost', 'port': 5432}}


In [ ]:
import re
from collections import Counter


def top_words(text, k):
    """
    题目描述
    实现函数 top_words(text, k)，统计文本中出现频率最高的前 k 个单词。
    要求：
    忽略大小写。
    只统计由字母和数字组成的单词。
    按出现次数降序排序；次数相同时按单词字典序升序排序。
    """
    # 提取所有符合条件的单词并转为小写
    words = re.findall(r"[a-zA-Z0-9]+", text.lower())

    # 统计词频
    word_counts = Counter(words)

    # 记录每个单词第一次出现的索引,按照单词出现的顺序进行排序
    first_index = {}
    for idx, word in enumerate(words):
        if word not in first_index:
            first_index[word] = idx

    # 按出现次数降序排序,次数相同时按单词字典序升序排序.
    sorted_words = sorted(
        word_counts.items(),
        key=lambda x: (-x[1], first_index[x[0]])
        )

    # 返回前 k 个
    return sorted_words[:k]


In [24]:
text = "Python is great. Python is simple, and python is powerful!"
print(top_words(text, 2))


[('python', 3), ('is', 3)]


In [26]:
text = "Hello Python Hello world Hello shanghai, Python is great. Python is simple, and python is powerful!"
print(top_words(text, 3))


[('python', 4), ('hello', 3), ('is', 3)]


In [ ]:
def group_by(records, key_func, agg_func):
    """
    题目描述
    实现函数 group_by(records, key_func, agg_func)，对字典列表进行分组聚合。
    key_func(record) 用于计算分组 key。
    agg_func(group_records) 用于对同组数据做聚合。
    返回一个字典，key 为分组 key，value 为聚合结果。
    """
    
    groups = {}
    for record in records:
        # 获取key值
        key = key_func(record)
        if key not in groups:
            groups[key] = []
        # 将记录添加到对应的组
        groups[key].append(record)

    result = {}
    for key, group_records in groups.items():
        result[key] = agg_func(group_records)

    return result


In [8]:
orders = [
    {"user": "A", "amount": 100},
    {"user": "B", "amount": 80},
    {"user": "A", "amount": 50},
    {"user": "C", "amount": 50},
]

result = group_by(
    orders,
    key_func=lambda r: r["user"],
    agg_func=lambda rows: sum(r["amount"] for r in rows),
)
print(result) 


{'A': 150, 'B': 80, 'C': 50}


In [ ]:

import time
import functools


def profile(func):
    """
    题目描述
    实现装饰器 profile(func)，用于统计函数执行耗时。
    要求：
    输出函数名、参数和执行耗时。
    被装饰函数返回值不能改变。
    保留原函数的 __name__ 和文档字符串。
    如果函数抛出异常，也需要输出耗时后再继续抛出异常。
    """
    
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            # 调用原函数并保存结果
            result = func(*args, **kwargs)
        except Exception as e:
            # 如果函数抛出异常，也需要输出耗时后再继续抛出异常
            elapsed = time.perf_counter() - start
            print(
                f"[消耗时长] {func.__name__}{args}{kwargs} 报错 {type(e).__name__}, 时长为: {elapsed:.6f}s"
            )
            raise
        else:
            # 计算耗时并输出
            elapsed = time.perf_counter() - start
            print(f"[消耗时长] {func.__name__}{args}{kwargs} 时长为: {elapsed:.6f}s")
            return result   

    return wrapper


In [37]:


from time import sleep


@profile
def add(a, b):
    """Return the sum of a and b."""
    sleep(0.3)  # 模拟耗时操作
    return a + b


result = add(1, 2)
assert result == 3, f"Expected 3, got {result}"



@profile
def greet(name, greeting="Hello"):
    """Greet someone."""
    return f"{greeting}, {name}!"


result = greet("Alice", greeting="Hi")
assert result == "Hi, Alice!"


assert add.__name__ == "add", f"__name__ was {add.__name__}"
assert add.__doc__ == "Return the sum of a and b."



@profile
def divide(a, b):
    return a / b


try:
    divide(1, 0)
except ZeroDivisionError:
    print("ZeroDivisionError was properly re-raised")  # 确认异常被重新抛出
else:
    assert False, "Should have raised ZeroDivisionError"



@profile
def say_hello():
    """No args."""
    return "hello"


result = say_hello()
assert result == "hello"


[消耗时长] add(1, 2){} 时长为: 0.30047930000000633s
[消耗时长] greet('Alice',){'greeting': 'Hi'} 时长为: 1.500004145782441e-06s
[消耗时长] divide(1, 0){} 报错 ZeroDivisionError, 时长为: 0.000002s
ZeroDivisionError was properly re-raised
[消耗时长] say_hello(){} 时长为: 5.00003807246685e-07s
